In [ ]:
# Import packages
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from spatialdata_io import visium
import os
import numpy as np
import anndata as ad
import igraph
import leidenalg

import json
from spatialdata import SpatialData
from spatialdata.models import Image2DModel
from spatialdata_io import visium
import spatialdata_plot

# Load the data

In [ ]:
# Set working directory
working_dir = os.getcwd()
data_path = working_dir


# Load the visium data (Healthy Mouse Liver_Mouse_001_Sample_001) into a spatial data object
sdata = visium(data_path, dataset_id = "GSM5764414_JBO001",counts_file = 'filtered_feature_bc_matrix.h5',tissue_positions_file = "tissue_positions_list.csv", scalefactors_file = "scalefactors_json.json",fullres_image_file=None)
adata = sdata.table
print(sdata)
adata.obs["total_counts"] = adata.X.sum(axis=1).A1 #The total number of counts of a certain gene.
adata.var["n_spots_by_counts"] = (adata.X > 0).sum(axis=0).A1 #The total number of spot that contains a certain gene.
adata.obs["n_genes_by_counts"] = (adata.X > 0).sum(axis=1).A1 #The total number of different genes that are expressed in a certain spot.

print(f'The original AnnData object contains: {adata.n_obs} spots and {adata.n_vars} genes. The original count is stored in AnnData.X')

# Load the annotation file of the mouse steady-state (StSt) Visium data (obtained from GSE192741)
annot = pd.read_csv('annot_mouseStStVisium.csv')

#select sample 'JBO1'
subset_annot = annot[annot['sample'] == 'JBO1'] 
subset_annot['spot'] = subset_annot['spot'].str[:-2]

In [ ]:
# Only select spot_ids in the spatial data object that are present in the annotation file
sdata.table.obs['spot_ind'] = sdata.table.obs.index
filtered_obs  = sdata.table.obs[sdata.table.obs['spot_ind'].isin(subset_annot['spot'])]
sdata_filtered = sdata.table[sdata.table.obs['spot_ind'].isin(subset_annot['spot']), :]
del sdata.tables['table']
sdata.table = sdata_filtered

# Only keep spot_ids in the shapes present in the annotation file
sdata.shapes['GSM5764414_JBO001']['spot_id'] = sdata.shapes['GSM5764414_JBO001'].index
filtered_shapes = sdata.shapes['GSM5764414_JBO001'][sdata.shapes['GSM5764414_JBO001']['spot_id'].isin(sdata.table.obs['spot_id'])]
del sdata.shapes['GSM5764414_JBO001']
sdata.shapes['GSM5764414_JBO001'] = filtered_shapes

sdata.pl.render_images(elements="GSM5764414_JBO001_lowres_image").pl.render_shapes(elements="GSM5764414_JBO001").pl.show(figsize = (8,8))

In [ ]:
sdata.pl.render_images().pl.render_shapes().pl.show("downscaled_hires")

central: CentralVeinEndothelialcells
mid: 
portal: PortalVeinEndothelialcells
periportal: 

In [ ]:
# read csv data
proportions = pd.read_csv("ex_vivo_proportions_rctd_liver_mouseVisium_JB01.csv",delimiter = '\t')
columns_to_select = ["CentralVeinEndothelialcells", "PortalVeinEndothelialcells","LSECs"]
prop = proportions[columns_to_select]

# add column based on the spatial zonation
proportions["spatial_zonation"] = prop.idxmax(axis=1)

# Reindex proportions to match the index with the index of the visium.table.obs
proportions.index = sdata.table.obs.index
subset_annot.index = sdata.table.obs.index

# Add the spatial_zonation to the annData object
sdata.table.obs['spatial_zonation'] = proportions['spatial_zonation']
sdata.table.obs['zonation_group'] = subset_annot['zonationGroup']
sdata.table.obs['CentralVeinEndothelialcells'] = proportions['CentralVeinEndothelialcells']
sdata.table.obs['PortalVeinEndothelialcells'] = proportions['PortalVeinEndothelialcells']

# Render image
sdata.pl.render_images(elements="GSM5764414_JBO001_hires_image").pl.render_shapes(elements="GSM5764414_JBO001", color="zonation_group").pl.show("downscaled_hires")
sdata.pl.render_images(elements="GSM5764414_JBO001_hires_image")\
           .pl.render_shapes(elements="GSM5764414_JBO001", color="n_genes_by_counts")\
           .pl.show("downscaled_hires", save="n_genes_by_counts.png")

# QC and preprocessing

## QC

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(8,5))

sns.histplot(adata.obs["total_counts"], kde=False, ax=axs[0,0])
axs[0,0].set_ylabel("Frequency")
axs[0,0].set_xlabel("Total Counts per Spot")
sns.histplot(
    adata.obs["total_counts"][adata.obs["total_counts"] < 15000],
    kde=False,
    bins=40,
    ax=axs[0,1],
)
axs[0,1].set_xlabel("Total Counts per Spot\n where total_counts < 15000")
axs[0,1].set_ylabel("Frequency")

sns.histplot(adata.obs["n_genes_by_counts"], kde=False, bins=60, ax=axs[1,0])
axs[1,0].set_xlabel("Number of different genes in a Spot")
axs[1,0].set_ylabel("Frequency")
sns.histplot(
    adata.obs["n_genes_by_counts"][adata.obs["n_genes_by_counts"] < 10000],
    kde=False,
    bins=60,
    ax=axs[1,1])
axs[1,1].set_xlabel("Number of different genes in a Spot\nwhere n_genes_by_counts < 10000")
axs[1,1].set_ylabel("Frequency")

plt.subplots_adjust(hspace=0.5)

fig, axs = plt.subplots(1, 1, figsize=(8,5))

sns.histplot(adata.obs["total_counts"], kde=False, ax=axs, color = 'lightcoral')

axs.set_ylabel("Frequency",fontsize=16)
axs.set_xlabel("Total counts per spot", fontsize=16)
axs.set_title("Histogram of Total Counts", fontsize=18) 
axs.tick_params(axis='both', labelsize=14)
axs.axvline(x=5000, color='black', linestyle='--', linewidth=2)
axs.axvline(x=55000, color = 'black',linestyle='--',linewidth=2)

## Preprocessing
Some spots have poor quality (e.g., damaged or stressed cells).

Expression of mitochondrial genes, total number of genes expressed, read distribution across genes and library size are good indicators for a spots quality.

A high-quality cell should have
- A high number of genes expressed
- A fair library size
- No major expression of mitochondrial genes, relative to other genes

In [ ]:
# Remove extremely high and low counts in one cell
sc.pp.filter_cells(adata, min_counts=5000) #e.g. empty droplets
sc.pp.filter_cells(adata, max_counts=55000) #e.g. multiple cells in one droplets

# Determine the percentage of mitochondrial genes per cell
mito_genes = adata.var_names[adata.var_names.str.startswith('mt')]
mito_counts = adata[:, mito_genes].X.sum(axis=1).A1
total_counts = adata.obs['total_counts'].values
adata.obs["mito_percentage"] = mito_counts/total_counts*100

# Remove cells with a high percentage of mitochondrial genes
adata = adata[adata.obs["mito_percentage"] < 20].copy()

# Filter non informative genes
sc.pp.filter_genes(adata, min_cells=10)

print(f"number of spots after filtering: {adata.n_obs}\nnumber of genes after filtering: {adata.n_vars}")

fig, axs = plt.subplots(2, 2, figsize=(8,5))

sns.histplot(adata.obs["total_counts"], kde=False, ax=axs[0,0])
axs[0,0].set_ylabel("Frequency")
axs[0,0].set_xlabel("Total Counts per Spot")
sns.histplot(
    adata.obs["total_counts"][adata.obs["total_counts"] < 15000],
    kde=False,
    bins=40,
    ax=axs[0,1],
)
axs[0,1].set_xlabel("Total Counts per Spot\n where total_counts < 15000")
axs[0,1].set_ylabel("Frequency")

sns.histplot(adata.obs["n_genes_by_counts"], kde=False, bins=60, ax=axs[1,0])
axs[1,0].set_xlabel("Number of different genes in a Spot")
axs[1,0].set_ylabel("Frequency")
sns.histplot(
    adata.obs["n_genes_by_counts"][adata.obs["n_genes_by_counts"] < 10000],
    kde=False,
    bins=60,
    ax=axs[1,1])
axs[1,1].set_xlabel("Number of different genes in a Spot\nwhere n_genes_by_counts < 10000")
axs[1,1].set_ylabel("Frequency")

plt.subplots_adjust(hspace=0.5)

fig, axs = plt.subplots(1, 1, figsize=(8,5))

sns.histplot(adata.obs["total_counts"], kde=False, ax=axs, color = 'lightcoral')
axs.set_ylabel("Frequency",fontsize=16)
axs.set_xlabel("Total counts per spot",fontsize=16)
axs.set_title("Histogram of total counts - filtered", fontsize=18) 
axs.tick_params(axis='both', labelsize=14)

### Normalisation

In [ ]:
sc.pp.normalize_total(adata, inplace=True)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, flavor="seurat", n_top_genes=2000)

# Clustering

In [ ]:
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.tl.leiden(adata, key_added="clusters", flavor="igraph", directed=False, n_iterations=2)
plt.rcParams["figure.figsize"] = (4, 4)
sc.pl.umap(adata, color=["total_counts", "n_genes_by_counts", "clusters"], wspace=0.4)

# Visualization in spatial coordinates

In [ ]:
visium.pl.render_images().pl.render_shapes().pl.show("downscaled_hires")
print(visium)

In [ ]:
visium["table"].to_df().sum(axis=0).sort_values(ascending=False).head(10)
#print(visium.table.obs["spot_id"])
data = visium.shapes
#print(data["GSM5764414_JBO001"])

print(visium.shapes['GSM5764414_JBO001']['geometry'])

## Deconvolution proportions

In [ ]:
# read csv data
proportions = pd.read_csv("ex_vivo_proportions_rctd_liver_mouseVisium_JB01.csv",delimiter = '\t')
columns_to_select = ["CentralVeinEndothelialcells", "Mesothelialcells", "PortalVeinEndothelialcells"]
prop = proportions[columns_to_select]

# add column based on the spatial zonation
proportions["spatial_zonation"] = prop.idxmax(axis=1)

print('len(proportions["spatial_zonation"]):',len(proportions["spatial_zonation"]))
print('len(visium.table.obs):',len(visium.table.obs))
print('len(visium.shapes[GSM5764414_JBO001]):',len(visium.shapes['GSM5764414_JBO001']))

#TO DO: only keep spots that are in Annotated_spots
#Annotated_spots = annot['cell'].astype(str)
#visium.table.obs['cell'] = visium.table.obs.index


#print(visium.table.obs[visium.table.obs['cell'].isin(Annotated_spots)])

#print(visium.table.obs['cell'])
#print(Annotated_spots)
#visium.table.obs["spatial_zonation"] = proportions["spatial_zonation"]

#visium.shapes['GSM5764414_JBO001']['spatial_zonation'] = 

visium.pl.render_images(elements="GSM5764414_JBO001_hires_image").pl.render_shapes(elements="GSM5764414_JBO001", color="spatial_zonation").pl.show()


### Cluster marker genes

In [ ]:
print(adata.obs.head())
import scanpy as sc

# PCA, Neighbors
sc.pp.pca(adata)
sc.pp.neighbors(adata)

# Leiden
sc.tl.leiden(adata, key_added="clusters")
print(adata.obs.head())
print(adata.obs["clusters"].unique())


In [ ]:
sc.tl.rank_genes_groups(adata, "clusters", method="t-test")
sc.pl.rank_genes_groups_heatmap(adata, groups="3", n_genes=10, groupby="clusters")